# Kafka Notebook

This notebook is a teaching-friendly walkthrough of a real Kafka streaming flow.
It is designed for classroom use and shows how a producer, a broker, and a consumer work together in a small but realistic pipeline.

## What learners will learn
1. How to start a local Kafka environment with Docker Compose
2. How a topic acts as the shared stream between producer and consumer
3. How to publish JSON events into Kafka
4. How to read those events back from Kafka
5. How to perform basic data-quality checks on a live stream
6. How schema evolution can be handled safely in a streaming system

## Before you begin
- Make sure Docker Desktop is running
- Make sure you have installed the Python dependencies from the requirements file
- Open this notebook from the Demo folder so the relative paths work correctly

> This notebook is intentionally simple and easy to follow. The goal is to make the Kafka concepts visible and understandable for first-time learners.

## Step 1 — Understand the demo architecture

A streaming pipeline has three main pieces:
- The producer: writes events into Kafka
- The broker: stores and distributes those events
- The consumer: reads events and processes them

In this notebook, we will simulate all three parts locally on your machine.
The producer will send JSON transaction events, the broker will hold them in a topic, and the consumer will read them back.

Why this matters:
- Kafka is often used for real-time data movement
- Producers and consumers are independent, which makes the system scalable
- A topic is the named stream that connects them

In [1]:
from pathlib import Path

workdir = Path.cwd()
print("Current working directory:", workdir)
print("Expected folder:", workdir.name)

Current working directory: C:\Shridhar\Study\BITS\Week 10\Demo
Expected folder: Demo


## Step 2 — Start the Kafka stack

This step starts the Kafka broker and the Kafka UI using Docker Compose.
The broker listens on port 9092, and the UI is exposed at http://localhost:8080.

Why this step matters:
- Kafka cannot work without a running broker
- Docker makes it easy to start a complete local environment quickly
- The UI gives a visual way to inspect topics and messages while the demo runs

In [3]:
import subprocess

result = subprocess.run(
    ["docker", "compose", "up", "-d"],
    cwd=str(Path.cwd()),
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("ERROR:")
    print(result.stderr)
else:
    print("Kafka stack started successfully.")


Kafka stack started successfully.


## Step 3 — Verify that the containers are running

After starting the stack, we should confirm that the broker and UI are available.
This is a useful habit when debugging infrastructure problems.

When the containers are running, you can also open the UI in your browser at http://localhost:8080.

In [4]:
result = subprocess.run(
    ["docker", "compose", "ps"],
    cwd=str(Path.cwd()),
    capture_output=True,
    text=True,
)
print(result.stdout)

NAME           IMAGE                           COMMAND                  SERVICE    CREATED          STATUS                    PORTS
lab-kafka      apache/kafka:latest             "/__cacert_entrypoinâ€¦"   kafka      19 seconds ago   Up 20 seconds (healthy)   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp
lab-kafka-ui   provectuslabs/kafka-ui:latest   "/bin/sh -c 'java --â€¦"   kafka-ui   19 seconds ago   Up 5 seconds              0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp



## Step 4 — Create a Kafka topic

A topic is the named stream where messages are stored.
Both the producer and consumer use the same topic name so they can communicate.

In this example, we create a topic named transactions.

Key idea:
- Topics are logical channels in Kafka
- Producers write into topics
- Consumers read from topics

In [5]:
import time
from kafka import KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

time.sleep(3)  # wait a little if the broker just started

bootstrap_servers = "localhost:9092"
topic = "transactions"

try:
    admin_client = KafkaAdminClient(
        bootstrap_servers=[bootstrap_servers],
        client_id="demo-notebook-admin"
    )

    try:
        admin_client.create_topics([NewTopic(name=topic, num_partitions=1, replication_factor=1)])
        print(f"Topic '{topic}' created.")
    except TopicAlreadyExistsError:
        print(f"Topic '{topic}' already exists.")
    finally:
        admin_client.close()

except Exception as e:
    print("Kafka admin error:", e)

Topic 'transactions' created.


## Step 5 — Produce sample transaction events

This is the producer part of the demo.
We will send several JSON events to the transactions topic.
Each event represents a transaction that might appear in a real banking or e-commerce system.

Each event contains:
- a unique transaction ID
- an account ID
- an amount
- a timestamp
- a merchant category

Why this is useful:
- It demonstrates the shape of a real event stream
- Learners can see that Kafka handles many small messages over time
- This is the foundation for later data-quality and monitoring concepts

In [6]:
import random
import time
import uuid
import json
from datetime import datetime, timezone
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=[bootstrap_servers],
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

accounts = [f"acct_{i:03d}" for i in range(1, 11)]
merchant_categories = ["grocery", "electronics", "travel", "dining", "fuel"]

for i in range(6):
    event = {
        "transaction_id": str(uuid.uuid4()),
        "account_id": random.choice(accounts),
        "amount": round(random.uniform(5, 250), 2),
        "event_time": datetime.now(timezone.utc).isoformat(),
        "merchant_category": random.choice(merchant_categories),
    }
    producer.send(topic, value=event)
    print(f"Sent {i+1}: {event}")
    time.sleep(0.2)

producer.flush()
producer.close()
print("Producer finished.")

Sent 1: {'transaction_id': '161480c0-db0d-4913-82c0-a201a0b6b4e8', 'account_id': 'acct_004', 'amount': 91.62, 'event_time': '2026-07-19T12:29:08.511300+00:00', 'merchant_category': 'dining'}
Sent 2: {'transaction_id': '7bde0ef3-04f4-44b9-b94c-8181fb9f7382', 'account_id': 'acct_001', 'amount': 221.81, 'event_time': '2026-07-19T12:29:08.844791+00:00', 'merchant_category': 'electronics'}
Sent 3: {'transaction_id': '262b8ec9-0c07-4055-a1a5-6adecc663f9d', 'account_id': 'acct_001', 'amount': 51.87, 'event_time': '2026-07-19T12:29:09.046216+00:00', 'merchant_category': 'grocery'}
Sent 4: {'transaction_id': '4e764743-1729-437e-ab12-d88cb14c387c', 'account_id': 'acct_001', 'amount': 135.63, 'event_time': '2026-07-19T12:29:09.249704+00:00', 'merchant_category': 'fuel'}
Sent 5: {'transaction_id': '99dc5043-ce50-4322-a062-7c4ef7570175', 'account_id': 'acct_007', 'amount': 229.34, 'event_time': '2026-07-19T12:29:09.452530+00:00', 'merchant_category': 'travel'}
Sent 6: {'transaction_id': 'fa721d62-a

## Step 6 — Read the messages back from Kafka

This is the consumer side of the demo.
We open a consumer, subscribe to the transactions topic, and read the messages that were just produced.

The key lesson here is that Kafka is not a queue for a single receiver only. It is a distributed log that many consumers can read from independently.

In [7]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    topic,
    bootstrap_servers=[bootstrap_servers],
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    auto_offset_reset="earliest",
    group_id="demo-notebook-group",
)

messages = []
for message in consumer:
    messages.append(message.value)
    print("Received:", message.value)
    if len(messages) >= 6:
        break

consumer.close()
print(f"Collected {len(messages)} messages.")

C:\Users\galan\AppData\Local\Temp\ipykernel_26688\926719661.py:3: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Received: {'transaction_id': '161480c0-db0d-4913-82c0-a201a0b6b4e8', 'account_id': 'acct_004', 'amount': 91.62, 'event_time': '2026-07-19T12:29:08.511300+00:00', 'merchant_category': 'dining'}
Received: {'transaction_id': '7bde0ef3-04f4-44b9-b94c-8181fb9f7382', 'account_id': 'acct_001', 'amount': 221.81, 'event_time': '2026-07-19T12:29:08.844791+00:00', 'merchant_category': 'electronics'}
Received: {'transaction_id': '262b8ec9-0c07-4055-a1a5-6adecc663f9d', 'account_id': 'acct_001', 'amount': 51.87, 'event_time': '2026-07-19T12:29:09.046216+00:00', 'merchant_category': 'grocery'}
Received: {'transaction_id': '4e764743-1729-437e-ab12-d88cb14c387c', 'account_id': 'acct_001', 'amount': 135.63, 'event_time': '2026-07-19T12:29:09.249704+00:00', 'merchant_category': 'fuel'}
Received: {'transaction_id': '99dc5043-ce50-4322-a062-7c4ef7570175', 'account_id': 'acct_007', 'amount': 229.34, 'event_time': '2026-07-19T12:29:09.452530+00:00', 'merchant_category': 'travel'}
Received: {'transaction_id':

## Step 7 — Apply a simple data-quality check

In real streaming systems, data is rarely perfect. Some records might be missing fields, some might have invalid values, and some might be duplicates.
This step introduces a tiny quality check so learners can see that data engineering is not only about moving data — it is also about validating it.

We check for:
- missing required fields
- non-positive amounts
- duplicate transaction IDs

This is the same kind of thinking used in real-time monitoring and data quality pipelines.

In [8]:
required_fields = ["transaction_id", "account_id", "amount", "event_time"]
seen_ids = set()
summary = {"total": 0, "missing_fields": 0, "invalid_values": 0, "duplicates": 0}

for event in messages:
    summary["total"] += 1
    missing = [field for field in required_fields if field not in event]
    if missing:
        summary["missing_fields"] += 1
        print("Missing fields:", missing)
        continue

    if event.get("amount", 0) <= 0:
        summary["invalid_values"] += 1
        print("Invalid amount:", event.get("amount"))

    tx_id = event.get("transaction_id")
    if tx_id in seen_ids:
        summary["duplicates"] += 1
        print("Duplicate transaction:", tx_id)
    else:
        seen_ids.add(tx_id)

print("\nQuality summary:")
print(json.dumps(summary, indent=2))


Quality summary:
{
  "total": 6,
  "missing_fields": 0,
  "invalid_values": 0,
  "duplicates": 0
}


## Step 8 — Show a simple schema evolution example

Real systems change over time. A new field might be added to a message format without breaking older consumers.
This is called schema evolution.

In this example, we create a new event that includes an additional field named device_fingerprint.
The older fields are still present, so the event remains compatible with the same basic consumer logic.

Why this matters:
- Systems evolve over time
- Producers and consumers do not always upgrade at the same time
- Backward-compatible changes are important in production streams

In [9]:
new_event = {
    "transaction_id": str(uuid.uuid4()),
    "account_id": "acct_005",
    "amount": 99.99,
    "event_time": datetime.now(timezone.utc).isoformat(),
    "merchant_category": "online",
    "device_fingerprint": "device_1234",
}

print("New event with optional field:")
print(json.dumps(new_event, indent=2))

New event with optional field:
{
  "transaction_id": "6fdbc383-6e9d-4bb5-8c05-6634c4d589ce",
  "account_id": "acct_005",
  "amount": 99.99,
  "event_time": "2026-07-19T12:29:44.106771+00:00",
  "merchant_category": "online",
  "device_fingerprint": "device_1234"
}


## Step 9 — Send the schema-evolved event

We publish the updated event to Kafka so learners can see that a producer can emit a slightly different message shape.
This demonstrates that a streaming system can evolve gradually rather than through a disruptive all-at-once migration.

In [10]:
producer = KafkaProducer(
    bootstrap_servers=[bootstrap_servers],
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)
producer.send(topic, value=new_event)
producer.flush()
producer.close()
print("Sent schema-evolved event.")

Sent schema-evolved event.


## Step 10 — Read the updated event back

The consumer can read the new event even though it contains an extra field.
This reinforces the idea that Kafka is flexible and that consuming code can be written to tolerate a changing schema.

In [11]:
consumer = KafkaConsumer(
    topic,
    bootstrap_servers=[bootstrap_servers],
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    auto_offset_reset="earliest",
    group_id="demo-notebook-group-read",
)

for message in consumer:
    if message.value.get("transaction_id") == new_event["transaction_id"]:
        print("Found the schema-evolved event:")
        print(json.dumps(message.value, indent=2))
        break
else:
    print("The schema-evolved event was not found in the current read window.")

consumer.close()

C:\Users\galan\AppData\Local\Temp\ipykernel_26688\4166778284.py:1: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Found the schema-evolved event:
{
  "transaction_id": "6fdbc383-6e9d-4bb5-8c05-6634c4d589ce",
  "account_id": "acct_005",
  "amount": 99.99,
  "event_time": "2026-07-19T12:29:44.106771+00:00",
  "merchant_category": "online",
  "device_fingerprint": "device_1234"
}


## Step 11 — Clean up the demo environment

When you are finished, stop the Kafka containers so your machine is left in a clean state.
This is especially helpful if you plan to push your work to GitHub or share the project with other learners.

In [1]:
print("To stop the demo stack, run this in a terminal:")
print("docker compose down -v")

To stop the demo stack, run this in a terminal:
docker compose down -v


---